# 다중 질의 검색(Multi-query retrieval)

하나의 사용자 질문을 여러 검색 관점으로 바꾸고, 각 질의의 결과를 합집합으로
결합합니다. 예전 `MultiQueryRetriever.from_llm()` 대신 Pydantic 구조화 출력과
Runnable 배치 호출을 사용해 생성 질의와 중복 제거 규칙을 명시합니다.


> **2026-09-21 업데이트**
>
> 이 노트북은 `langchain 1.4.2`, `langchain-core 1.6.3`,
> `langchain-openai 1.6.2`, `langchain-chroma 1.1.0` 기준으로 다시 작성했습니다.
> LangChain v1에서 예전 `langchain.retrievers` 구현은 `langchain-classic`으로
> 이동했고 `langchain-community`도 보관 상태이므로, 새 코드에서는 두 패키지와
> `langchain-teddynote`에 의존하지 않습니다. 대신 `langchain-core`의 Runnable,
> 공급자별 파트너 패키지, 명시적인 검색 함수를 조합합니다.
>
> 공식 참고: [LangChain v1 변경 사항](https://docs.langchain.com/oss/python/releases/langchain-v1),
> [v1 마이그레이션](https://docs.langchain.com/oss/python/migrate/langchain-v1),
> [OpenAI 임베딩](https://docs.langchain.com/oss/python/integrations/embeddings/openai),
> [Chroma 통합](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)


In [ ]:
# 최초 1회만 주석을 해제하세요.
# %pip install -qU "langchain==1.4.2" "langchain-core==1.6.3" \
#   "langchain-openai==1.6.2" "langchain-chroma==1.1.0" \
#   "langchain-text-splitters==1.1.2" python-dotenv httpx beautifulsoup4


In [ ]:
import getpass
import os
from uuid import uuid4

import httpx
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

CHROMA_CONFIGURATION = {"hnsw": {"space": "cosine"}}


## 웹 문서 로드와 벡터 인덱스

웹 로딩은 검색 기법의 핵심이 아니므로 표준 HTTP 클라이언트와 BeautifulSoup로
최소 구현합니다. JavaScript 렌더링 페이지나 운영 크롤러에는 전용 수집기를 쓰세요.


In [ ]:
url = "https://teddylee777.github.io/openai/openai-assistant-tutorial/"
response = httpx.get(url, follow_redirects=True, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")
for tag in soup(["script", "style", "nav", "footer"]):
    tag.decompose()
page_text = "\n".join(
    line.strip() for line in soup.get_text("\n").splitlines() if line.strip()
)

source_doc = Document(page_content=page_text, metadata={"source": url})
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    add_start_index=True,
)
chunks = splitter.split_documents([source_doc])
chunks = [
    Document(
        page_content=chunk.page_content,
        metadata={**chunk.metadata, "doc_id": f"chunk-{index}"},
    )
    for index, chunk in enumerate(chunks)
]

vectorstore = Chroma.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name=f"multi-query-{uuid4().hex}",
    ids=[chunk.metadata["doc_id"] for chunk in chunks],
    collection_configuration=CHROMA_CONFIGURATION,
)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"인덱싱한 청크 수: {len(chunks)}")


## 구조화된 질의 생성


In [ ]:
class QueryVariants(BaseModel):
    queries: list[str] = Field(
        min_length=3,
        max_length=5,
        description="원 질문과 의미는 같지만 검색 관점이 다른 한국어 질의",
    )


prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "벡터 검색의 표현 편향을 줄이기 위해 서로 다른 관점의 검색 질의를 "
            "3~5개 생성하세요. 답을 만들지 말고 질의만 생성하세요.",
        ),
        ("user", "원 질문: {question}"),
    ]
)
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
model = init_chat_model(f"openai:{model_name}", temperature=0)
query_generator = prompt | model.with_structured_output(QueryVariants)

question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."
generated = query_generator.invoke({"question": question})
generated.queries


## 병렬 검색과 안정적인 중복 제거

원 질문도 검색 목록에 포함합니다. 텍스트가 우연히 같은 문서를 합치지 않도록
인덱싱 때 부여한 `doc_id`를 중복 제거 키로 사용합니다.


In [ ]:
def multi_query_search(request: dict) -> dict:
    original = request["question"]
    variants = query_generator.invoke({"question": original}).queries
    queries = list(dict.fromkeys([original, *variants]))
    result_lists = base_retriever.batch(
        queries,
        config={"max_concurrency": request.get("max_concurrency", 5)},
    )

    unique_docs: dict[str, Document] = {}
    for documents in result_lists:
        for doc in documents:
            unique_docs.setdefault(doc.metadata["doc_id"], doc)
    return {"queries": queries, "documents": list(unique_docs.values())}


multi_query_retriever = RunnableLambda(multi_query_search).with_config(
    {"run_name": "multi_query_retriever"}
)
result = multi_query_retriever.invoke({"question": question})

print("생성·사용된 질의:")
for query in result["queries"]:
    print("-", query)
print(f"\n고유 문서 수: {len(result['documents'])}")
print(result["documents"][0].page_content)
